# BioHack 2026 — Combined Colab Notebook

This is the single-notebook, Colab-friendly version of this repo's three notebooks
(`01_data_ingest_and_qc.ipynb`, `02_classical_clustering_and_annotation.ipynb`,
`03_geneformer_foundation_model.ipynb`), concatenated in order:

- **Part 1 — Data Ingest & Quality Control**
- **Part 2 — Classical Clustering & Cell Type Annotation**
- **Part 3 — A Single-Cell Foundation Model for Cell Type Classification**

Opening the three original notebooks separately in Colab each spins up its own fresh
runtime, so Part 2 can't find the data Part 1 wrote, packages have to be reinstalled, etc.
This notebook exists so you can open **one file** and run straight through — everything
below (repo clone, install, Geneformer checkpoint download, kernel restart) only needs to
happen once, right here, at the top.

If you're running locally instead of on Colab, use the three separate notebooks under
`notebooks/` and follow `README.md` — the Colab-specific setup below is unnecessary there.

## Colab setup — run these cells once, in order

1. **Pin the runtime to Python 3.12** (Command Palette → "runtime version" → fallback/previous
   runtime). Colab's latest default runtime has moved to Python 3.13, and this repo's
   `pyproject.toml` requires `>=3.10,<3.13` — `pip install -e .` fails outright on 3.13.
2. Clone the repo and install its pinned dependencies (`scanpy`, `torch`, `transformers`, etc.)
   in one environment.
3. Download the Geneformer checkpoint (~50MB, one-time) needed for Part 3.
4. **Restart the runtime.** Colab's base image ships NumPy 2.x; this repo pins `numpy<2`
   (needed for `torch==2.2.x` ABI compatibility), and installing NumPy 1.x into an
   already-running Colab kernel leaves stale NumPy 2.x compiled extensions loaded, which
   crashes with `ValueError: numpy.dtype size changed, may indicate binary incompatibility`.
   Restarting starts a clean process against the newly-installed NumPy.

See `README.md` in the repo root for the full writeup of all of this.

In [ ]:
import sys

print("Python version:", sys.version)
if sys.version_info[:2] >= (3, 13):
    print(
        "\n\u26a0\ufe0f  This repo requires Python 3.10-3.12 (not 3.13+). "
        "Open the Command Palette (Cmd/Ctrl+Shift+P), search 'runtime version', "
        "select the fallback/previous runtime, reconnect, then rerun this cell."
    )

In [ ]:
# Clone the repo and install its pinned dependencies. This single environment covers
# everything -- classical scanpy analysis *and* the Geneformer notebook (see README.md
# "Setup" for why NumPy is pinned <2 and why numba/llvmlite are pinned to exact versions).
!git clone https://github.com/aneesav/biohack-2026.git
%cd biohack-2026
!pip install -e . -q

# The original notebooks use paths like "../data/..." relative to notebooks/, so land there
# and stay there for the rest of this notebook.
%cd notebooks

In [ ]:
# One-time download of the Geneformer-V1-10M checkpoint (~50MB) needed for Part 3.
# Geneformer isn't on PyPI -- it's distributed from its Hugging Face repo alongside much
# larger checkpoints (V2 models are 100M-300M+ params); this script uses huggingface_hub's
# allow_patterns to fetch only the V1-10M weights + library code.
!bash ../scripts/setup_geneformer.sh

# You generally don't need a Hugging Face token for this (Geneformer-V1-10M is public).
# If you hit a rate limit, set HF_TOKEN before rerunning the line above:
# import os
# os.environ["HF_TOKEN"] = "hf_your_token_here"

### Restart the runtime now

Run the cell below -- it will kill and restart the Colab kernel (this is expected and
required, see the explanation above). Colab will show a "session crashed" notice; ignore it.

**After it restarts, do not rerun the cells above.** Everything on disk (the cloned repo,
installed packages, downloaded checkpoint) persists across the restart -- just continue from
the cell after the restart cell.

In [ ]:
import os

os.kill(os.getpid(), 9)  # restarts the Colab kernel

**Continue here after the restart.** The cell below just re-establishes the working
directory (a fresh kernel process starts back at `/content`, not `notebooks/`).

In [ ]:
import os

if os.path.basename(os.getcwd()) != "notebooks":
    for candidate in ("biohack-2026/notebooks", "notebooks"):
        if os.path.isdir(candidate):
            os.chdir(candidate)
            break
print("Working directory:", os.getcwd())

---

# Part 1 — Data Ingest & Quality Control

Single-cell RNA-seq (scRNA-seq) measures gene expression in thousands of individual cells at
once. The raw output is a big, sparse, noisy matrix: **cells x genes**, where each entry is a
transcript count. Before any analysis, we need to know which cells are real, high-quality
measurements and which are technical artifacts (empty droplets, doublets, dying cells).

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white")

## 1. Load the data

We load raw (unfiltered, unnormalized) counts. `adata.X` is the cells x genes count matrix,
`adata.obs` holds per-cell metadata, `adata.var` holds per-gene metadata.

In [ ]:
adata = sc.read_h5ad("../data/pbmc3k_raw.h5ad")
adata.raw = adata  # keep an untouched copy of the raw counts for later
adata

In [ ]:
# Peek at the matrix itself: cells x genes, mostly zeros (a typical scRNA-seq sparsity pattern)
print(f"{adata.n_obs} cells x {adata.n_vars} genes")
print(f"matrix sparsity: {1 - adata.X.nnz / (adata.n_obs * adata.n_vars):.3f} fraction zero")
adata.X[:5, :10].toarray()

## 2. Compute QC metrics

Three numbers matter most for a first pass on scRNA-seq QC:

- **`total_counts`** — total transcripts captured per cell. Too low usually means an empty droplet
  or degraded cell; suspiciously high can mean two cells were captured together (a "doublet").
- **`n_genes_by_counts`** — number of distinct genes detected per cell. Correlates with
  `total_counts` but catches different failure modes.
- **`pct_counts_mt`** — percent of counts coming from mitochondrial genes (`MT-` prefix in human).
  Dying or membrane-compromised cells leak cytoplasmic mRNA and retain mitochondrial mRNA, so high
  mitochondrial fraction is a classic low-quality-cell signal.

In [ ]:
# Flag mitochondrial genes (human gene naming convention)
adata.var["mt"] = adata.var_names.str.startswith("MT-")

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    percent_top=None,
    log1p=False,
    inplace=True,
)
adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe()

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts")
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt")

**Stop and look at the plots.** Where would you draw the line for "too few genes detected"?
For "too much mitochondrial content"? There's no universally correct threshold — it depends on the
tissue, the protocol, and how conservative you want to be. We'll use thresholds that are standard
for PBMC data, but in your own work you should set these by looking at *your* distributions, not by
copying numbers from a tutorial.

## 3. Filter low-quality cells

We'll drop cells with too few or implausibly many detected genes (the latter can indicate
doublets), and cells with high mitochondrial content.

In [ ]:
n_before = adata.n_obs

adata = adata[adata.obs.n_genes_by_counts > 200, :]
adata = adata[adata.obs.n_genes_by_counts < 2000, :]
adata = adata[adata.obs.pct_counts_mt < 5, :].copy()

print(f"{n_before} cells -> {adata.n_obs} cells after QC filtering "
      f"({n_before - adata.n_obs} removed)")

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

## 4. Save the QC'd data

We save this filtered-but-not-yet-normalized AnnData so later notebooks can pick up from here
without re-running QC. Notebook 02 will normalize and cluster it; notebook 03 (the foundation model
notebook) needs the **raw counts** of these same cells, which is why we keep `adata.raw` intact
rather than overwriting `adata.X` with normalized values here.

In [ ]:
adata.write("../data/pbmc3k_qc.h5ad")
print("Saved", adata.shape)

## Recap

- scRNA-seq data is noisy and needs QC before anything else — garbage in, garbage out applies
  doubly to ML on top of it.
- `total_counts`, `n_genes_by_counts`, and `pct_counts_mt` are the standard first-pass QC metrics.
- Thresholds are dataset-specific judgment calls, not universal constants.

**Next:** `02_classical_clustering_and_annotation.ipynb` — normalize this data, cluster it, and
assign cell type labels using classical methods (PCA, Leiden clustering, marker genes).

---

# Part 2 — Classical Clustering & Cell Type Annotation

Now that we have QC'd data, we'll run the standard scRNA-seq analysis pipeline: normalize,
find the genes that actually vary across cells, reduce dimensionality, cluster, and figure out
what each cluster *is* biologically. This is the "classical" (pre-foundation-model) workflow
that Part 3 will benchmark a pretrained model against.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white")

adata = sc.read_h5ad("../data/pbmc3k_qc.h5ad")
adata

## 1. Normalize and transform

Raw counts aren't directly comparable across cells — a cell with more total RNA will have higher
counts for every gene, which has nothing to do with biology. We normalize each cell to the same
total count, then log-transform (expression data is heavily right-skewed; log-transforming makes
variance more comparable across the expression range, which most downstream methods assume).

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

## 2. Find highly variable genes

Most genes are either uninformative housekeeping genes or just noise at this depth of sequencing.
Restricting to highly variable genes (HVGs) focuses the analysis on genes that actually distinguish
cell populations, and makes PCA much more meaningful.

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
print(f"{adata.var.highly_variable.sum()} highly variable genes out of {adata.n_vars}")
sc.pl.highly_variable_genes(adata)

In [ ]:
adata.raw = adata  # save the full log-normalized matrix before subsetting to HVGs
adata = adata[:, adata.var.highly_variable].copy()

# Scale each gene to zero mean, unit variance, clipping extreme outliers
sc.pp.scale(adata, max_value=10)

## 3. PCA, neighbors, UMAP

PCA compresses thousands of genes into a handful of components that capture most of the
cell-to-cell variation. We then build a nearest-neighbor graph in PCA space and use it both for
clustering (Leiden) and for a 2D visualization (UMAP).

In [ ]:
sc.tl.pca(adata, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)

## 4. Leiden clustering

Leiden clustering finds densely-connected communities in the neighbor graph — groups of cells that
look more like each other than like the rest of the dataset. It does **not** know what a "cell
type" is; it just finds structure. We still have to interpret what each cluster represents.

In [ ]:
sc.tl.leiden(adata, resolution=0.5, flavor="igraph", n_iterations=2)
sc.pl.umap(adata, color=["leiden"], title="Leiden Clustering")

**Stop and look at the UMAP.** PCA + Leiden separates cell populations, but the cluster labels
("0", "1", "2"...) are meaningless on their own. To know whether cluster 0 is T cells or B cells, we
need to look at *which genes* distinguish each cluster, and compare them against genes we already
know mark specific cell types — that's the next step.

## 5. Rank marker genes per cluster

For each cluster, we statistically test which genes are most differentially expressed compared to
all other clusters. The top of that ranked list is a cluster's marker genes.

`rank_genes_groups` defaults to using `adata.raw` when it's set, which is the full
log-normalized gene set we saved before subsetting to HVGs above — so marker gene ranking isn't
limited to only the genes that passed the HVG filter.

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="t-test")
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False)

In [ ]:
top_markers = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(10)
top_markers

## 6. Manual annotation using canonical markers

This is where domain knowledge comes in. PBMCs have well-characterized marker genes from decades of
immunology research, e.g.:

| Marker gene(s) | Cell type |
|---|---|
| `CD3D`, `CD3E`, `IL32` | T cells |
| `CD8A` | CD8+ T cells |
| `NKG7`, `GNLY`, `GZMA` | NK cells |
| `CD79A`, `MS4A1`, `CD74` | B cells |
| `LYZ`, `S100A9`, `FTL` | Monocytes |
| `FCER1G`, `HLA-DPA1` | Dendritic cells |
| `PPBP`, `PF4` | Megakaryocytes / platelets |

Look at the top markers table above and the ranked-genes plot, match them against this table, and
fill in the mapping below. **The cluster numbers and best-matching cell type will vary depending on
the exact data and resolution** — look at your own output rather than assuming the dictionary below
is already correct for your run.

In [ ]:
# EDIT THIS based on what you see in the marker gene plot above
cluster_annotations = {
    "0": "CD4 T cells",
    "1": "NK cells / Cytotoxic T cells",
    "2": "B cells",
    "3": "Monocytes",
    "4": "Dendritic cells",
    "5": "Megakaryocytes",
}

adata.obs["cell_type"] = adata.obs["leiden"].map(cluster_annotations).astype("category")
sc.pl.umap(adata, color=["cell_type"], title="Manually Annotated Cell Types")

## 7. Save labeled data

We save the cell type labels (and the underlying PCA features) — notebook 03 will treat these
manual annotations as ground truth to evaluate against, and will compare PCA features against
foundation model embeddings on a downstream classification task.

In [ ]:
labels = adata.obs[["leiden", "cell_type"]].copy()
labels["cell_id"] = labels.index
labels.to_csv("../data/pbmc3k_cell_type_labels.csv", index=False)

pca_df = pd.DataFrame(adata.obsm["X_pca"], index=adata.obs_names)
pca_df.to_csv("../data/pbmc3k_pca_features.csv")

print("Saved labels for", len(labels), "cells across", labels.cell_type.nunique(), "cell types")
labels.cell_type.value_counts()

## Recap

- Classical scRNA-seq analysis: normalize -> HVGs -> PCA -> neighbor graph -> Leiden clustering ->
  marker genes -> manual annotation against known biology.
- Every step up to clustering is unsupervised and "knows" nothing about cell type. The actual
  labeling step depends entirely on a human matching marker genes against prior literature.
- This means classical annotation doesn't scale well: every new dataset, tissue, or species needs
  someone who already knows the relevant marker genes.

**Next:** `03_geneformer_foundation_model.ipynb` — replace the "human matches markers against
known biology" step with a pretrained single-cell foundation model, and quantitatively compare how
well its learned representations support cell type classification versus the classical PCA
features above, especially when very few labeled cells are available.

---

# Part 3 — A Single-Cell Foundation Model for Cell Type Classification

In Part 2, annotating cell types required a human who already knew the relevant marker genes
for the tissue in question. That doesn't scale, and it doesn't transfer to a new tissue or a
new dataset. Here we use a pretrained single-cell foundation model (Geneformer-V1-10M) and ask:
how much less labeled data does it need to match classical features at cell type
classification? The Geneformer checkpoint was already downloaded in the Colab setup section
above, so we skip straight to loading the data.

In [ ]:
import pickle
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib.pyplot as plt
from datasets import load_from_disk
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

GENEFORMER_REPO = Path("../vendor/geneformer_repo")
assert GENEFORMER_REPO.exists(), (
    "Geneformer not found — run `bash scripts/setup_geneformer.sh` from the repo root first."
)

from geneformer import TranscriptomeTokenizer

## 1. Prepare the data

Geneformer needs **raw, unnormalized counts** across the cell's full transcriptome (not the
log-normalized, HVG-subsetted matrix from notebook 02) — it does its own rank-based normalization
internally. We recover that from `adata.raw`, which notebook 01 set before any filtering of genes
or normalization.

We'll also pull in the cell type labels from notebook 02, and subsample to a manageable number of
cells per type so this runs in about a minute — bump `N_PER_TYPE` up (or remove the subsampling
entirely) if you have more time and want embeddings for every cell.

In [ ]:
adata = sc.read_h5ad("../data/pbmc3k_qc.h5ad")
raw = adata.raw.to_adata()  # full gene panel, raw counts, only cell-level QC filtering applied

labels = pd.read_csv("../data/pbmc3k_cell_type_labels.csv").set_index("cell_id")
raw = raw[raw.obs_names.isin(labels.index)].copy()
raw.obs["cell_type"] = labels.loc[raw.obs_names, "cell_type"].values

N_PER_TYPE = 120
rng = np.random.default_rng(0)
keep = []
for ct, grp in raw.obs.groupby("cell_type"):
    idx = grp.index.tolist()
    n = min(N_PER_TYPE, len(idx))
    keep.extend(rng.choice(idx, size=n, replace=False).tolist())
raw = raw[keep].copy()

print(raw.shape)
raw.obs["cell_type"].value_counts()

## 2. Map gene symbols to Ensembl IDs

Geneformer's vocabulary is keyed by Ensembl gene ID, not gene symbol. Conveniently, the package
ships its own symbol-to-Ensembl-ID dictionary (the same one it used during pretraining), so this is
a pure local lookup.

Not every gene symbol in our data will have a match: Geneformer's vocabulary covers ~25-40K
protein-coding and well-characterized genes, while PBMC3k's gene panel includes thousands of
obscure, non-coding, or poorly annotated loci that were never going to be in scope for a model
trained on curated single-cell atlases. Seeing roughly half your genes drop out here is expected,
not a bug.

In [ ]:
with open(GENEFORMER_REPO / "geneformer/gene_dictionaries_30m/gene_name_id_dict_gc30M.pkl", "rb") as f:
    symbol_to_ensembl = pickle.load(f)

raw.var["ensembl_id"] = [symbol_to_ensembl.get(g) for g in raw.var_names]
raw.obs["n_counts"] = np.asarray(raw.X.sum(axis=1)).flatten()
raw.obs["cell_id"] = raw.obs_names

n_mapped = raw.var["ensembl_id"].notna().sum()
print(f"mapped {n_mapped} / {raw.n_vars} genes to Ensembl IDs")

## 3. Tokenize

`TranscriptomeTokenizer` does the rank-value encoding described above: for each cell, it normalizes
expression against the gene median dictionary from Geneformer's pretraining corpus, ranks genes
accordingly, and writes out a token sequence per cell as a Hugging Face `datasets.Dataset`.

In [ ]:
work_dir = Path("/tmp/geneformer_work")
shutil.rmtree(work_dir, ignore_errors=True)
(work_dir / "in").mkdir(parents=True)
raw.write(work_dir / "in" / "cells.h5ad")

tokenizer = TranscriptomeTokenizer(
    custom_attr_name_dict={"cell_id": "cell_id"},
    nproc=1,
    model_version="V1",     # the 10M-parameter checkpoint was pretrained as a "V1" model
    special_token=False,    # V1 models predate the <cls>/<eos> special tokens V2 uses
    model_input_size=2048,  # V1's context length
)
tokenizer.tokenize_data(work_dir / "in", work_dir / "out", "cells", file_format="h5ad")

dataset = load_from_disk(work_dir / "out" / "cells.dataset")
print(f"tokenized {len(dataset)} cells")
print("example token sequence length:", dataset[0]["length"])

## 4. Load the model and extract cell embeddings

A note on the implementation here: Geneformer ships a convenience class for this (`EmbExtractor`),
but as of this writing its embedding-extraction code path has `device="cuda"` hardcoded with no CPU
or Apple Silicon (MPS) fallback — it'll crash with `AssertionError: Torch not compiled with CUDA
enabled` on any machine without an Nvidia GPU, which is most laptops at a hackathon. This is a good
reminder that real research tooling has real rough edges. Instead, we load the underlying
Hugging Face model directly and write our own short, transparent, device-agnostic forward pass —
which also makes it clearer what's actually happening: we mean-pool the model's last hidden layer
across each cell's (unpadded) token sequence to get one vector per cell.

One efficiency detail: cells have wildly different numbers of expressed genes, so token sequences
range from a few hundred to ~2,000 tokens. Batching cells of very different lengths together forces
heavy padding (and on Apple's MPS backend, batches with extreme padding can stall badly). Sorting by
sequence length before batching — then unsorting the results back to the original order — fixes
both problems and is a generally useful trick whenever you're batching variable-length sequences.

In [ ]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print("using device:", device)

from transformers import BertForMaskedLM

model = BertForMaskedLM.from_pretrained(
    GENEFORMER_REPO / "Geneformer-V1-10M", output_hidden_states=True
)
model.eval().to(device)

with open(GENEFORMER_REPO / "geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl", "rb") as f:
    token_dictionary = pickle.load(f)
pad_token_id = token_dictionary["<pad>"]

In [ ]:
order = sorted(range(len(dataset)), key=lambda i: dataset[i]["length"])
dataset_sorted = dataset.select(order)

batch_size = 16
embeddings_sorted = []

t0 = time.time()
with torch.no_grad():
    for i in range(0, len(dataset_sorted), batch_size):
        batch = dataset_sorted[i : i + batch_size]
        lengths = batch["length"]
        max_len = max(lengths)

        input_ids = torch.tensor(
            [seq + [pad_token_id] * (max_len - len(seq)) for seq in batch["input_ids"]],
            device=device,
        )
        attention_mask = torch.tensor(
            [[1] * l + [0] * (max_len - l) for l in lengths], device=device
        )

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.hidden_states[-1]               # (batch, seq_len, hidden_dim)
        mask = attention_mask.unsqueeze(-1).bool()
        mean_pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
        embeddings_sorted.append(mean_pooled.cpu().numpy())

embeddings_sorted = np.concatenate(embeddings_sorted)
inverse_order = np.argsort(order)
embeddings = embeddings_sorted[inverse_order]            # back to dataset's original cell order
cell_ids = dataset["cell_id"]

print(f"extracted {embeddings.shape[0]} embeddings of dimension {embeddings.shape[1]} "
      f"in {time.time() - t0:.1f}s")

np.save("../data/pbmc3k_geneformer_embeddings.npy", embeddings)
pd.Series(cell_ids).to_csv("../data/pbmc3k_geneformer_cell_ids.csv", index=False)

## 5. Visualize: does the foundation model's embedding space separate cell types?

Geneformer never saw our `cell_type` labels — they come entirely from notebook 02's classical
pipeline. If the embedding space below visibly separates by color, that's a real signal the
pretrained model learned biologically meaningful structure with zero supervision on this dataset.

In [ ]:
import anndata as ad

emb_adata = ad.AnnData(
    X=embeddings,
    obs=pd.DataFrame({"cell_type": pd.Categorical(
        labels.loc[cell_ids, "cell_type"].values
    )}, index=cell_ids),
)
sc.pp.neighbors(emb_adata, use_rep="X")
sc.tl.umap(emb_adata)
sc.pl.umap(emb_adata, color="cell_type", title="Geneformer embeddings (UMAP)")

## 6. The real test: few-shot cell type classification

A UMAP plot is suggestive but qualitative. Here's a concrete, quantitative comparison: train a
simple logistic regression classifier to predict `cell_type` from features, using only a handful of
labeled examples per class — then check accuracy on the rest. We run this for both:

- **Classical features**: the PCA components from notebook 02's pipeline.
- **Geneformer embeddings**: this notebook's foundation model representations.

This simulates the most common real constraint in biology: unlabeled cells are cheap (sequence
anything), but *expert-annotated* cells are expensive and scarce. A representation that needs fewer
labeled examples to reach good accuracy is directly valuable.

In [ ]:
pca_features = pd.read_csv("../data/pbmc3k_pca_features.csv", index_col=0)
gf_features = pd.DataFrame(embeddings, index=cell_ids)

common_cells = gf_features.index.intersection(pca_features.index).intersection(labels.index)
y = labels.loc[common_cells, "cell_type"].values
X_pca = pca_features.loc[common_cells].values
X_gf = gf_features.loc[common_cells].values

print(f"{len(common_cells)} cells with both feature sets and labels")

In [ ]:
def few_shot_accuracy(X, y, n_shots, n_seeds=15):
    accs = []
    for seed in range(n_seeds):
        r = np.random.default_rng(seed * 100 + n_shots)
        train_idx = []
        for cls in np.unique(y):
            cls_idx = np.where(y == cls)[0]
            if len(cls_idx) <= n_shots:
                continue
            train_idx.extend(r.choice(cls_idx, size=n_shots, replace=False))
        train_idx = np.array(train_idx)
        test_idx = np.array([i for i in range(len(y)) if i not in set(train_idx)])

        scaler = StandardScaler().fit(X[train_idx])
        X_train, X_test = scaler.transform(X[train_idx]), scaler.transform(X[test_idx])

        clf = LogisticRegression(max_iter=2000)
        clf.fit(X_train, y[train_idx])
        accs.append(clf.score(X_test, y[test_idx]))
    return np.mean(accs), np.std(accs)

shot_counts = [1, 3, 5, 10, 20]
results = []
for n_shots in shot_counts:
    pca_mean, pca_std = few_shot_accuracy(X_pca, y, n_shots)
    gf_mean, gf_std = few_shot_accuracy(X_gf, y, n_shots)
    results.append((n_shots, pca_mean, pca_std, gf_mean, gf_std))
    print(f"{n_shots:>2} shots/class -- PCA: {pca_mean:.3f} +/- {pca_std:.3f}  "
          f"Geneformer: {gf_mean:.3f} +/- {gf_std:.3f}")

In [ ]:
results_df = pd.DataFrame(
    results, columns=["n_shots", "pca_mean", "pca_std", "gf_mean", "gf_std"]
)

plt.figure(figsize=(7, 5))
plt.errorbar(results_df.n_shots, results_df.pca_mean, yerr=results_df.pca_std,
             marker="o", label="Classical PCA features", capsize=3)
plt.errorbar(results_df.n_shots, results_df.gf_mean, yerr=results_df.gf_std,
             marker="o", label="Geneformer embeddings", capsize=3)
plt.xlabel("labeled examples per cell type (\"shots\")")
plt.ylabel("held-out classification accuracy")
plt.title("Few-shot cell type classification: foundation model vs. classical features")
plt.legend()
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.show()

**Stop and look at the curve.** In our run, Geneformer embeddings reached roughly 80% accuracy
with a *single* labeled cell per type, while classical PCA features needed around 10-20 labeled
cells per type to catch up. That gap is the entire value proposition of a foundation model in this
setting: pretraining on ~30M cells gave the model a strong prior about what distinguishes cell
types *before it ever saw this dataset*, so it needs far less task-specific labeled data to do well.

The two methods converge as labeled data increases — with enough labels, the classical features
eventually catch up. That's the expected, realistic story: foundation models are most valuable
exactly when labels are scarce, not a strictly-better-in-all-cases replacement for classical
analysis.

Some questions worth discussing with your team:

- Why might the gap be largest at very low shot counts and shrink as shots increase?
- We used a 10M-parameter checkpoint pretrained on ~30M cells. Geneformer's largest public
  checkpoint has 316M parameters and was pretrained on ~104M cells. What would you predict happens
  to this curve with a bigger model?
- We froze the embeddings and only trained a linear classifier on top. What might change if we
  fine-tuned the whole model instead (and what would that cost in compute/time)?
- This dataset's classes are all well-separated, common PBMC types. Would you expect the same size
  of advantage for a harder problem — e.g., distinguishing two closely related T cell subtypes?

## Recap

- A foundation model pretrained on millions of cells can encode a strong, general-purpose prior
  about cell biology — usable for a new dataset's classification task without any task-specific
  pretraining, just a lightweight classifier on top of its embeddings.
- That prior is most valuable exactly when labeled data is scarce, which is the normal situation in
  real biological annotation work.
- Real ML/bio tooling has rough edges (the hardcoded-CUDA issue here) that you'll hit constantly in
  practice — reading the library's source when something breaks is a normal and necessary part of
  the work, not a sign you're doing something wrong.

This closes the loop on the whole pipeline: raw counts -> QC -> classical clustering and manual
annotation -> foundation model embeddings -> a quantitative, reproducible argument for when and why
the foundation model is worth using.